⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks Migration: SILOS_SIL_INVENTORYPRODUCTDIMENSION\n",
        "\n",
        "**Conversion Timestamp**: 2024-07-30 12:00:00\n",
        "\n",
        "This notebook contains the Spark SQL conversion of the ODI process for `SILOS_SIL_INVENTORYPRODUCTDIMENSION`. It handles incremental updates and inserts into the `W_INVENTORY_PRODUCT_D` target table, including pre-merge category updates, and manages ETL load dates.\n",
        "\n",
        "**Key transformations applied:**\n",
        "- Oracle-specific syntax (`NVL`, `DECODE`, `SYSDATE`, `VARCHAR2`, `NUMBER`, `DATE`, `UROWID`, `ROWID`, `||`, `BEGIN...END`, `DROP ... PURGE`, `NOLOGGING`, `NOCOMPRESS`, `/*+ append */`, `CREATE INDEX`) replaced with Spark SQL equivalents.\n",
        "- `MERGE` statements constructed from separate `UPDATE` and `INSERT` logic.\n",
        "- Schema and table names converted to `workspace.schema_lowercase.table_lowercase`.\n",
        "- ODI parameters converted to Databricks widgets.\n",
        "- Oracle `ROWID` checks in `CASE` statements converted to `JOIN` conditions or PK existence checks.\n",
        "- Oracle sequences (`NEXTVAL`) removed for assumed `IDENTITY` columns.\n",
        "- Timestamp format strings adjusted for Spark SQL.\n",
        "\n",
        "**Manual Actions Required (Post-conversion validation):**\n",
        "1.  **Widget Values**: Ensure default values for `ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `WH_DATASOURCE_NUM_ID`, `ETL_USAGE_CODE`, `LOW_DATE`, `SOURCE_CODE`, `TARGET_CODE`, `ETL_PROC_WID`, `EXECUTION_ID`, `PRUNE_DAYS`, `IS_INCREMENTAL`, `ODI_SESS_NO` are correctly set for the specific environment.\n",
        "2.  **Target Table DDL**: Verify the DDL for `workspace.prxbi_dw.w_inventory_product_d` has `ROW_WID` as `BIGINT GENERATED ALWAYS AS IDENTITY` to align with the removal of `NEXTVAL`.\n",
        "3.  **Pre-requisite Tables**: Ensure `w_ora_invitem_category_tmp` and `w_inventory_product_ds` (and other lookup tables like `w_prod_cat_dh`, `w_busn_location_d`, etc.) are created and populated correctly upstream as part of the overall ETL flow if not defined within this notebook.\n",
        "4.  **Error Logging**: The original ODI script had extensive error logging comments, but it was set to 'Non Error Logging Mode'. The `E$_` table is created, but error capture logic (e.g., `INSERT INTO E$_...`) is not present in the provided source. If error logging is needed, this part needs to be implemented.\n",
        "5.  **ZORDER BY**: Consider adding `OPTIMIZE ... ZORDER BY` statements on large target tables for performance if key columns are identified after data analysis."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"999\", \"DATASOURCE_NUM_ID (BIAPPS)\")\n",
        "dbutils.widgets.text(\"WH_DATASOURCE_NUM_ID\", \"999\", \"WH_DATASOURCE_NUM_ID (BIAPPS)\")\n",
        "dbutils.widgets.text(\"ETL_USAGE_CODE\", \"__NOT_APPLICABLE__\", \"ETL_USAGE_CODE (BIAPPS)\")\n",
        "dbutils.widgets.text(\"LOW_DATE\", \"1900-01-01 00:00:00\", \"LOW_DATE (BIAPPS)\")\n",
        "dbutils.widgets.text(\"SOURCE_CODE\", \"DEFAULT\", \"SOURCE_CODE (BIAPPS)\")\n",
        "dbutils.widgets.text(\"TARGET_CODE\", \"DEFAULT\", \"TARGET_CODE (BIAPPS)\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"0\", \"ETL_PROC_WID (BIAPPS)\")\n",
        "dbutils.widgets.text(\"EXECUTION_ID\", \"0\", \"EXECUTION_ID (BIAPPS)\")\n",
        "dbutils.widgets.text(\"PRUNE_DAYS\", \"-1\", \"PRUNE_DAYS (BIAPPS)\")\n",
        "dbutils.widgets.text(\"IS_INCREMENTAL\", \"Y\", \"IS_INCREMENTAL (BIAPPS)\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"0\", \"ODI Session Number\")"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {1}: Get ETL load status\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_load_status AS\n",
        "SELECT\n",
        "    CASE\n",
        "        WHEN COUNT(*) > 0 THEN 'Y'\n",
        "        ELSE 'N'\n",
        "    END AS load_status_exists\n",
        "FROM\n",
        "    workspace.prxbi_dw.w_etl_load_dates\n",
        "WHERE\n",
        "    package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "    AND (\n",
        "        datasource_num_id = ${DATASOURCE_NUM_ID}\n",
        "        OR datasource_num_id = ${WH_DATASOURCE_NUM_ID}\n",
        "    )\n",
        "    AND etl_usage_code = '${ETL_USAGE_CODE}'\n",
        "    AND committed = '1';"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "code",
      "source": [
        "display(spark.sql(\"SELECT 'DATASOURCE_NUM_ID' AS Parameter, '${DATASOURCE_NUM_ID}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'WH_DATASOURCE_NUM_ID' AS Parameter, '${WH_DATASOURCE_NUM_ID}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'ETL_USAGE_CODE' AS Parameter, '${ETL_USAGE_CODE}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'LOW_DATE' AS Parameter, '${LOW_DATE}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'SOURCE_CODE' AS Parameter, '${SOURCE_CODE}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'TARGET_CODE' AS Parameter, '${TARGET_CODE}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'ETL_PROC_WID' AS Parameter, '${ETL_PROC_WID}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'EXECUTION_ID' AS Parameter, '${EXECUTION_ID}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'PRUNE_DAYS' AS Parameter, '${PRUNE_DAYS}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'IS_INCREMENTAL' AS Parameter, '${IS_INCREMENTAL}' AS Value UNION ALL\"))\n",
        "display(spark.sql(\"SELECT 'ODI_SESS_NO' AS Parameter, '${ODI_SESS_NO}' AS Value\"));\n",
        "\n",
        "display(spark.sql(\"SELECT * FROM v_etl_load_status;\"))"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Error Tables"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {60}: Drop error table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.e_inventory_product_d;"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {70}: Create error table\n",
        "CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.e_inventory_product_d (\n",
        "    ora_err_number$         BIGINT,\n",
        "    ora_err_mesg$           STRING,\n",
        "    ora_err_rowid$          STRING,\n",
        "    ora_err_optyp$          STRING,\n",
        "    ora_err_tag$            STRING,\n",
        "    ind_update              STRING,\n",
        "    diagnostic_rowid        STRING,\n",
        "    error_type_ind          STRING,\n",
        "    autocorrect_ind         STRING  DEFAULT 'N',\n",
        "    autocorrect_code        STRING,\n",
        "    autocorrect_desc        STRING,\n",
        "    committed               STRING  DEFAULT '0',\n",
        "    row_wid                 STRING,\n",
        "    product_wid             STRING,\n",
        "    inventory_org_wid       STRING,\n",
        "    plant_loc_wid           STRING,\n",
        "    product_num             STRING,\n",
        "    abc_ind                 STRING,\n",
        "    planner_code            STRING,\n",
        "    procurement_type_code   STRING,\n",
        "    spc_proc_type_code      STRING,\n",
        "    buyer_code              STRING,\n",
        "    buyer_name              STRING,\n",
        "    commodity_code          STRING,\n",
        "    commodity_uom_code      STRING,\n",
        "    profit_center_num       STRING,\n",
        "    reorder_point           STRING,\n",
        "    safety_stock_level      STRING,\n",
        "    min_lot_size            STRING,\n",
        "    max_lot_size            STRING,\n",
        "    fixed_lot_size          STRING,\n",
        "    max_stock_level         STRING,\n",
        "    lot_ordering_cost       STRING,\n",
        "    mrp_time_fence          STRING,\n",
        "    ext_procure_time        STRING,\n",
        "    internal_mfg_time       STRING,\n",
        "    max_storage_days        STRING,\n",
        "    mrp_profile_code        STRING,\n",
        "    mrp_type_code           STRING,\n",
        "    mrp_grp_code            STRING,\n",
        "    lot_size_code           STRING,\n",
        "    backflush_ind           STRING,\n",
        "    qa_inspect_ind          STRING,\n",
        "    repetitive_mfg_ind      STRING,\n",
        "    bulk_item_ind           STRING,\n",
        "    forecast_period         STRING,\n",
        "    mfg_uom_code            STRING,\n",
        "    issue_uom_code          STRING,\n",
        "    manufacturing_place     STRING,\n",
        "    loading_type_code       STRING,\n",
        "    int_store_loc_code      STRING,\n",
        "    ext_store_loc_code      STRING,\n",
        "    active_flg              STRING,\n",
        "    created_by_wid          STRING,\n",
        "    changed_by_wid          STRING,\n",
        "    created_on_dt           STRING,\n",
        "    changed_on_dt           STRING,\n",
        "    aux1_changed_on_dt      STRING,\n",
        "    aux2_changed_on_dt      STRING,\n",
        "    aux3_changed_on_dt      STRING,\n",
        "    aux4_changed_on_dt      STRING,\n",
        "    src_eff_from_dt         STRING,\n",
        "    src_eff_to_dt           STRING,\n",
        "    effective_from_dt       STRING,\n",
        "    effective_to_dt         STRING,\n",
        "    current_flg             STRING,\n",
        "    w_insert_dt             STRING,\n",
        "    w_update_dt             STRING,\n",
        "    datasource_num_id       STRING,\n",
        "    etl_proc_wid            STRING,\n",
        "    integration_id          STRING,\n",
        "    tenant_id               STRING,\n",
        "    x_custom                STRING,\n",
        "    inv_prod_cat1           STRING,\n",
        "    inv_prod_cat2           STRING,\n",
        "    inv_prod_cat3           STRING,\n",
        "    inv_prod_cat4           STRING,\n",
        "    inv_prod_cat5           STRING,\n",
        "    inv_prod_cat6           STRING,\n",
        "    inv_prod_cat7           STRING,\n",
        "    inv_prod_cat8           STRING,\n",
        "    inv_prod_cat9           STRING,\n",
        "    inv_prod_cat10          STRING,\n",
        "    inv_prod_cat1_wid       STRING,\n",
        "    inv_prod_cat2_wid       STRING,\n",
        "    inv_prod_cat3_wid       STRING,\n",
        "    inv_prod_cat4_wid       STRING,\n",
        "    inv_prod_cat5_wid       STRING,\n",
        "    inv_prod_cat6_wid       STRING,\n",
        "    inv_prod_cat7_wid       STRING,\n",
        "    inv_prod_cat8_wid       STRING,\n",
        "    inv_prod_cat9_wid       STRING,\n",
        "    inv_prod_cat10_wid      STRING,\n",
        "    invoiceable_item_flag   STRING,\n",
        "    invoice_enabled_flag    STRING,\n",
        "    primary_uom_code        STRING,\n",
        "    c_primary_uom_code      STRING,\n",
        "    unspsc_code             STRING,\n",
        "    unspsc_inv_prod_cat_wid STRING,\n",
        "    commodity_name          STRING,\n",
        "    commodity_uom_name      STRING,\n",
        "    ext_store_loc_name      STRING,\n",
        "    int_store_loc_name      STRING,\n",
        "    issue_uom_name          STRING,\n",
        "    loading_type_name       STRING,\n",
        "    lot_size_name           STRING,\n",
        "    mfg_uom_name            STRING,\n",
        "    mrp_grp_name            STRING,\n",
        "    mrp_profile_name        STRING,\n",
        "    mrp_type_name           STRING,\n",
        "    planner_name            STRING,\n",
        "    primary_uom_name        STRING,\n",
        "    procurement_type_name   STRING,\n",
        "    profit_center_name      STRING,\n",
        "    spc_proc_type_name      STRING,\n",
        "    status_code             STRING,\n",
        "    w_status_code           STRING,\n",
        "    product_type_code       STRING,\n",
        "    make_buy_ind            STRING,\n",
        "    fixed_lead_time         STRING,\n",
        "    variable_lead_time      STRING,\n",
        "    cumulative_total_lead_time STRING,\n",
        "    postprocessing_lead_time STRING,\n",
        "    preprocessing_lead_time STRING,\n",
        "    process_quality_enabled_flg STRING,\n",
        "    x_price_sequence        STRING,\n",
        "    x_organization_name     STRING,\n",
        "    x_product_desc          STRING,\n",
        "    x_uom_desc              STRING,\n",
        "    x_inv_item_flg          STRING,\n",
        "    x_stock_item_flg        STRING,\n",
        "    x_trans_flg             STRING,\n",
        "    x_rev_flg               STRING,\n",
        "    x_cost_flg              STRING,\n",
        "    x_gcoa_acct             STRING,\n",
        "    x_gcoa_prod             STRING,\n",
        "    x_tax_cat               STRING,\n",
        "    organization_id         STRING,\n",
        "    x_gcoa_loc_acct         STRING,\n",
        "    delete_flg              STRING\n",
        ") USING DELTA;"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Flow Table"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {110}: Drop flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_inventory_product_d_flow;"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {120}: Create flow table\n",
        "CREATE TABLE workspace.prxbi_dw.i_inventory_product_d_flow (\n",
        "    src_eff_from_dt         TIMESTAMP,\n",
        "    datasource_num_id       BIGINT,\n",
        "    integration_id          STRING,\n",
        "    row_wid                 BIGINT,\n",
        "    product_wid             BIGINT,\n",
        "    inventory_org_wid       BIGINT,\n",
        "    plant_loc_wid           BIGINT,\n",
        "    product_num             STRING,\n",
        "    abc_ind                 STRING,\n",
        "    planner_code            STRING,\n",
        "    procurement_type_code   STRING,\n",
        "    spc_proc_type_code      STRING,\n",
        "    buyer_code              STRING,\n",
        "    buyer_name              STRING,\n",
        "    commodity_code          STRING,\n",
        "    commodity_uom_code      STRING,\n",
        "    profit_center_num       STRING,\n",
        "    reorder_point           DOUBLE,\n",
        "    safety_stock_level      DOUBLE,\n",
        "    min_lot_size            DOUBLE,\n",
        "    max_lot_size            DOUBLE,\n",
        "    fixed_lot_size          DOUBLE,\n",
        "    max_stock_level         DOUBLE,\n",
        "    lot_ordering_cost       DOUBLE,\n",
        "    mrp_time_fence          DOUBLE,\n",
        "    ext_procure_time        DOUBLE,\n",
        "    internal_mfg_time       DOUBLE,\n",
        "    max_storage_days        DOUBLE,\n",
        "    mrp_profile_code        STRING,\n",
        "    mrp_type_code           STRING,\n",
        "    mrp_grp_code            STRING,\n",
        "    lot_size_code           STRING,\n",
        "    backflush_ind           STRING,\n",
        "    qa_inspect_ind          STRING,\n",
        "    repetitive_mfg_ind      STRING,\n",
        "    bulk_item_ind           STRING,\n",
        "    forecast_period         STRING,\n",
        "    mfg_uom_code            STRING,\n",
        "    issue_uom_code          STRING,\n",
        "    manufacturing_place     STRING,\n",
        "    loading_type_code       STRING,\n",
        "    int_store_loc_code      STRING,\n",
        "    ext_store_loc_code      STRING,\n",
        "    active_flg              STRING,\n",
        "    created_by_wid          BIGINT,\n",
        "    changed_by_wid          BIGINT,\n",
        "    created_on_dt           TIMESTAMP,\n",
        "    changed_on_dt           TIMESTAMP,\n",
        "    aux1_changed_on_dt      TIMESTAMP,\n",
        "    aux2_changed_on_dt      TIMESTAMP,\n",
        "    aux3_changed_on_dt      TIMESTAMP,\n",
        "    aux4_changed_on_dt      TIMESTAMP,\n",
        "    src_eff_to_dt           TIMESTAMP,\n",
        "    effective_from_dt       TIMESTAMP,\n",
        "    effective_to_dt         TIMESTAMP,\n",
        "    delete_flg              STRING,\n",
        "    current_flg             STRING,\n",
        "    w_insert_dt             TIMESTAMP,\n",
        "    w_update_dt             TIMESTAMP,\n",
        "    etl_proc_wid            BIGINT,\n",
        "    tenant_id               STRING,\n",
        "    x_custom                STRING,\n",
        "    inv_prod_cat1           STRING,\n",
        "    inv_prod_cat2           STRING,\n",
        "    inv_prod_cat3           STRING,\n",
        "    inv_prod_cat4           STRING,\n",
        "    inv_prod_cat5           STRING,\n",
        "    inv_prod_cat6           STRING,\n",
        "    inv_prod_cat7           STRING,\n",
        "    inv_prod_cat8           STRING,\n",
        "    inv_prod_cat9           STRING,\n",
        "    inv_prod_cat10          STRING,\n",
        "    inv_prod_cat1_wid       BIGINT,\n",
        "    inv_prod_cat2_wid       BIGINT,\n",
        "    inv_prod_cat3_wid       BIGINT,\n",
        "    inv_prod_cat4_wid       BIGINT,\n",
        "    inv_prod_cat5_wid       BIGINT,\n",
        "    inv_prod_cat6_wid       BIGINT,\n",
        "    inv_prod_cat7_wid       BIGINT,\n",
        "    inv_prod_cat8_wid       BIGINT,\n",
        "    inv_prod_cat9_wid       BIGINT,\n",
        "    inv_prod_cat10_wid      BIGINT,\n",
        "    invoiceable_item_flag   STRING,\n",
        "    invoice_enabled_flag    STRING,\n",
        "    primary_uom_code        STRING,\n",
        "    c_primary_uom_code      STRING,\n",
        "    unspsc_code             STRING,\n",
        "    unspsc_inv_prod_cat_wid BIGINT,\n",
        "    commodity_name          STRING,\n",
        "    commodity_uom_name      STRING,\n",
        "    ext_store_loc_name      STRING,\n",
        "    int_store_loc_name      STRING,\n",
        "    issue_uom_name          STRING,\n",
        "    loading_type_name       STRING,\n",
        "    lot_size_name           STRING,\n",
        "    mfg_uom_name            STRING,\n",
        "    mrp_grp_name            STRING,\n",
        "    mrp_profile_name        STRING,\n",
        "    mrp_type_name           STRING,\n",
        "    planner_name            STRING,\n",
        "    primary_uom_name        STRING,\n",
        "    procurement_type_name   STRING,\n",
        "    profit_center_name      STRING,\n",
        "    spc_proc_type_name      STRING,\n",
        "    status_code             STRING,\n",
        "    w_status_code           STRING,\n",
        "    product_type_code       STRING,\n",
        "    make_buy_ind            STRING,\n",
        "    fixed_lead_time         DOUBLE,\n",
        "    variable_lead_time      DOUBLE,\n",
        "    cumulative_total_lead_time DOUBLE,\n",
        "    postprocessing_lead_time DOUBLE,\n",
        "    preprocessing_lead_time DOUBLE,\n",
        "    process_quality_enabled_flg STRING,\n",
        "    x_price_sequence        STRING,\n",
        "    x_organization_name     STRING,\n",
        "    x_product_desc          STRING,\n",
        "    x_uom_desc              STRING,\n",
        "    x_inv_item_flg          STRING,\n",
        "    x_stock_item_flg        STRING,\n",
        "    x_trans_flg             STRING,\n",
        "    x_rev_flg               STRING,\n",
        "    x_cost_flg              STRING,\n",
        "    x_gcoa_acct             STRING,\n",
        "    x_gcoa_prod             STRING,\n",
        "    x_tax_cat               STRING,\n",
        "    organization_id         STRING,\n",
        "    x_gcoa_loc_acct         STRING,\n",
        "    ind_update              STRING\n",
        ") USING DELTA;"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {130}: Insert into flow table\n",
        "INSERT INTO workspace.prxbi_dw.i_inventory_product_d_flow (\n",
        "    product_wid,\n",
        "    inventory_org_wid,\n",
        "    plant_loc_wid,\n",
        "    product_num,\n",
        "    abc_ind,\n",
        "    planner_code,\n",
        "    procurement_type_code,\n",
        "    spc_proc_type_code,\n",
        "    buyer_code,\n",
        "    buyer_name,\n",
        "    commodity_code,\n",
        "    commodity_uom_code,\n",
        "    profit_center_num,\n",
        "    reorder_point,\n",
        "    safety_stock_level,\n",
        "    min_lot_size,\n",
        "    max_lot_size,\n",
        "    fixed_lot_size,\n",
        "    max_stock_level,\n",
        "    lot_ordering_cost,\n",
        "    mrp_time_fence,\n",
        "    ext_procure_time,\n",
        "    internal_mfg_time,\n",
        "    max_storage_days,\n",
        "    mrp_profile_code,\n",
        "    mrp_type_code,\n",
        "    mrp_grp_code,\n",
        "    lot_size_code,\n",
        "    backflush_ind,\n",
        "    qa_inspect_ind,\n",
        "    repetitive_mfg_ind,\n",
        "    bulk_item_ind,\n",
        "    forecast_period,\n",
        "    mfg_uom_code,\n",
        "    issue_uom_code,\n",
        "    manufacturing_place,\n",
        "    loading_type_code,\n",
        "    int_store_loc_code,\n",
        "    ext_store_loc_code,\n",
        "    active_flg,\n",
        "    created_by_wid,\n",
        "    changed_by_wid,\n",
        "    created_on_dt,\n",
        "    changed_on_dt,\n",
        "    aux1_changed_on_dt,\n",
        "    aux2_changed_on_dt,\n",
        "    aux3_changed_on_dt,\n",
        "    aux4_changed_on_dt,\n",
        "    src_eff_from_dt,\n",
        "    src_eff_to_dt,\n",
        "    effective_from_dt,\n",
        "    delete_flg,\n",
        "    datasource_num_id,\n",
        "    integration_id,\n",
        "    tenant_id,\n",
        "    x_custom,\n",
        "    inv_prod_cat1,\n",
        "    inv_prod_cat2,\n",
        "    inv_prod_cat3,\n",
        "    inv_prod_cat4,\n",
        "    inv_prod_cat5,\n",
        "    inv_prod_cat6,\n",
        "    inv_prod_cat7,\n",
        "    inv_prod_cat8,\n",
        "    inv_prod_cat9,\n",
        "    inv_prod_cat10,\n",
        "    inv_prod_cat1_wid,\n",
        "    inv_prod_cat2_wid,\n",
        "    inv_prod_cat3_wid,\n",
        "    inv_prod_cat4_wid,\n",
        "    inv_prod_cat5_wid,\n",
        "    inv_prod_cat6_wid,\n",
        "    inv_prod_cat7_wid,\n",
        "    inv_prod_cat8_wid,\n",
        "    inv_prod_cat9_wid,\n",
        "    inv_prod_cat10_wid,\n",
        "    invoiceable_item_flag,\n",
        "    invoice_enabled_flag,\n",
        "    primary_uom_code,\n",
        "    c_primary_uom_code,\n",
        "    unspsc_code,\n",
        "    unspsc_inv_prod_cat_wid,\n",
        "    commodity_name,\n",
        "    commodity_uom_name,\n",
        "    ext_store_loc_name,\n",
        "    int_store_loc_name,\n",
        "    issue_uom_name,\n",
        "    loading_type_name,\n",
        "    lot_size_name,\n",
        "    mfg_uom_name,\n",
        "    mrp_grp_name,\n",
        "    mrp_profile_name,\n",
        "    mrp_type_name,\n",
        "    planner_name,\n",
        "    primary_uom_name,\n",
        "    procurement_type_name,\n",
        "    profit_center_name,\n",
        "    spc_proc_type_name,\n",
        "    status_code,\n",
        "    w_status_code,\n",
        "    product_type_code,\n",
        "    make_buy_ind,\n",
        "    fixed_lead_time,\n",
        "    variable_lead_time,\n",
        "    cumulative_total_lead_time,\n",
        "    postprocessing_lead_time,\n",
        "    preprocessing_lead_time,\n",
        "    process_quality_enabled_flg,\n",
        "    x_price_sequence,\n",
        "    x_organization_name,\n",
        "    x_product_desc,\n",
        "    x_uom_desc,\n",
        "    x_inv_item_flg,\n",
        "    x_stock_item_flg,\n",
        "    x_trans_flg,\n",
        "    x_rev_flg,\n",
        "    x_cost_flg,\n",
        "    x_gcoa_acct,\n",
        "    x_gcoa_prod,\n",
        "    x_tax_cat,\n",
        "    organization_id,\n",
        "    x_gcoa_loc_acct,\n",
        "    current_flg,\n",
        "    effective_to_dt,\n",
        "    ind_update\n",
        ")\n",
        "SELECT\n",
        "    C.product_wid,\n",
        "    C.inventory_org_wid,\n",
        "    C.plant_loc_wid,\n",
        "    C.product_num,\n",
        "    C.abc_ind,\n",
        "    C.planner_code,\n",
        "    C.procurement_type_code,\n",
        "    C.spc_proc_type_code,\n",
        "    C.buyer_code,\n",
        "    C.buyer_name,\n",
        "    C.commodity_code,\n",
        "    C.commodity_uom_code,\n",
        "    C.profit_center_num,\n",
        "    C.reorder_point,\n",
        "    C.safety_stock_level,\n",
        "    C.min_lot_size,\n",
        "    C.max_lot_size,\n",
        "    C.fixed_lot_size,\n",
        "    C.max_stock_level,\n",
        "    C.lot_ordering_cost,\n",
        "    C.mrp_time_fence,\n",
        "    C.ext_procure_time,\n",
        "    C.internal_mfg_time,\n",
        "    C.max_storage_days,\n",
        "    C.mrp_profile_code,\n",
        "    C.mrp_type_code,\n",
        "    C.mrp_grp_code,\n",
        "    C.lot_size_code,\n",
        "    C.backflush_ind,\n",
        "    C.qa_inspect_ind,\n",
        "    C.repetitive_mfg_ind,\n",
        "    C.bulk_item_ind,\n",
        "    C.forecast_period,\n",
        "    C.mfg_uom_code,\n",
        "    C.issue_uom_code,\n",
        "    C.manufacturing_place,\n",
        "    C.loading_type_code,\n",
        "    C.int_store_loc_code,\n",
        "    C.ext_store_loc_code,\n",
        "    C.active_flg,\n",
        "    C.created_by_wid,\n",
        "    C.changed_by_wid,\n",
        "    C.created_on_dt,\n",
        "    C.changed_on_dt,\n",
        "    C.aux1_changed_on_dt,\n",
        "    C.aux2_changed_on_dt,\n",
        "    C.aux3_changed_on_dt,\n",
        "    C.aux4_changed_on_dt,\n",
        "    C.src_eff_from_dt,\n",
        "    C.src_eff_to_dt,\n",
        "    C.effective_from_dt,\n",
        "    C.delete_flg,\n",
        "    C.datasource_num_id,\n",
        "    C.integration_id,\n",
        "    C.tenant_id,\n",
        "    C.x_custom,\n",
        "    C.inv_prod_cat1,\n",
        "    C.inv_prod_cat2,\n",
        "    C.inv_prod_cat3,\n",
        "    C.inv_prod_cat4,\n",
        "    C.inv_prod_cat5,\n",
        "    C.inv_prod_cat6,\n",
        "    C.inv_prod_cat7,\n",
        "    C.inv_prod_cat8,\n",
        "    C.inv_prod_cat9,\n",
        "    C.inv_prod_cat10,\n",
        "    C.inv_prod_cat1_wid,\n",
        "    C.inv_prod_cat2_wid,\n",
        "    C.inv_prod_cat3_wid,\n",
        "    C.inv_prod_cat4_wid,\n",
        "    C.inv_prod_cat5_wid,\n",
        "    C.inv_prod_cat6_wid,\n",
        "    C.inv_prod_cat7_wid,\n",
        "    C.inv_prod_cat8_wid,\n",
        "    C.inv_prod_cat9_wid,\n",
        "    C.inv_prod_cat10_wid,\n",
        "    C.invoiceable_item_flag,\n",
        "    C.invoice_enabled_flag,\n",
        "    C.primary_uom_code,\n",
        "    C.c_primary_uom_code,\n",
        "    C.unspsc_code,\n",
        "    C.unspsc_inv_prod_cat_wid,\n",
        "    C.commodity_name,\n",
        "    C.commodity_uom_name,\n",
        "    C.ext_store_loc_name,\n",
        "    C.int_store_loc_name,\n",
        "    C.issue_uom_name,\n",
        "    C.loading_type_name,\n",
        "    C.lot_size_name,\n",
        "    C.mfg_uom_name,\n",
        "    C.mrp_grp_name,\n",
        "    C.mrp_profile_name,\n",
        "    C.mrp_type_name,\n",
        "    C.planner_name,\n",
        "    C.primary_uom_name,\n",
        "    C.procurement_type_name,\n",
        "    C.profit_center_name,\n",
        "    C.spc_proc_type_name,\n",
        "    C.status_code,\n",
        "    C.w_status_code,\n",
        "    C.product_type_code,\n",
        "    C.make_buy_ind,\n",
        "    C.fixed_lead_time,\n",
        "    C.variable_lead_time,\n",
        "    C.cumulative_total_lead_time,\n",
        "    C.postprocessing_lead_time,\n",
        "    C.preprocessing_lead_time,\n",
        "    C.process_quality_enabled_flg,\n",
        "    C.x_price_sequence,\n",
        "    C.x_organization_name,\n",
        "    C.x_product_desc,\n",
        "    C.x_uom_desc,\n",
        "    C.x_inv_item_flg,\n",
        "    C.x_stock_item_flg,\n",
        "    C.x_trans_flg,\n",
        "    C.x_rev_flg,\n",
        "    C.x_cost_flg,\n",
        "    C.x_gcoa_acct,\n",
        "    C.x_gcoa_prod,\n",
        "    C.x_tax_cat,\n",
        "    C.organization_id,\n",
        "    C.x_gcoa_loc_acct,\n",
        "    'Y',\n",
        "    to_timestamp('3714-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss'),\n",
        "    CASE\n",
        "        WHEN T.integration_id IS NOT NULL\n",
        "        AND (\n",
        "            T.changed_on_dt = C.changed_on_dt OR (T.changed_on_dt IS NULL AND C.changed_on_dt IS NULL)\n",
        "        )\n",
        "        AND (\n",
        "            T.aux1_changed_on_dt = C.aux1_changed_on_dt OR (T.aux1_changed_on_dt IS NULL AND C.aux1_changed_on_dt IS NULL)\n",
        "        )\n",
        "        AND (\n",
        "            T.aux2_changed_on_dt = C.aux2_changed_on_dt OR (T.aux2_changed_on_dt IS NULL AND C.aux2_changed_on_dt IS NULL)\n",
        "        )\n",
        "        AND (\n",
        "            T.aux3_changed_on_dt = C.aux3_changed_on_dt OR (T.aux3_changed_on_dt IS NULL AND C.aux3_changed_on_dt IS NULL)\n",
        "        )\n",
        "        AND (\n",
        "            T.aux4_changed_on_dt = C.aux4_changed_on_dt OR (T.aux4_changed_on_dt IS NULL AND C.aux4_changed_on_dt IS NULL)\n",
        "        )\n",
        "        THEN 'N'\n",
        "        WHEN T.integration_id IS NOT NULL THEN 'U'\n",
        "        ELSE 'I'\n",
        "    END\n",
        "FROM\n",
        "    (\n",
        "        SELECT\n",
        "            COALESCE(INLINE_VIEW.scd1_wid_1, 0)                                                         AS product_wid,\n",
        "            COALESCE(INLINE_VIEW.scd1_wid, 0)                                                           AS inventory_org_wid,\n",
        "            COALESCE(INLINE_VIEW.row_wid, 0)                                                            AS plant_loc_wid,\n",
        "            INLINE_VIEW.product_num                                                                     AS product_num,\n",
        "            INLINE_VIEW.abc_ind                                                                         AS abc_ind,\n",
        "            COALESCE(INLINE_VIEW.planner_code, '__NOT_APPLICABLE__')                                    AS planner_code,\n",
        "            COALESCE(INLINE_VIEW.procurement_type_code, '__NOT_APPLICABLE__')                           AS procurement_type_code,\n",
        "            COALESCE(INLINE_VIEW.spc_proc_type_code, '__NOT_APPLICABLE__')                              AS spc_proc_type_code,\n",
        "            COALESCE(INLINE_VIEW.buyer_code, '__NOT_APPLICABLE__')                                      AS buyer_code,\n",
        "            INLINE_VIEW.buyer_name                                                                      AS buyer_name,\n",
        "            COALESCE(INLINE_VIEW.commodity_code, '__NOT_APPLICABLE__')                                  AS commodity_code,\n",
        "            COALESCE(INLINE_VIEW.commodity_uom_code, '__NOT_APPLICABLE__')                              AS commodity_uom_code,\n",
        "            INLINE_VIEW.profit_center_num                                                               AS profit_center_num,\n",
        "            INLINE_VIEW.reorder_point                                                                   AS reorder_point,\n",
        "            INLINE_VIEW.safety_stock_level                                                              AS safety_stock_level,\n",
        "            INLINE_VIEW.min_lot_size                                                                    AS min_lot_size,\n",
        "            INLINE_VIEW.max_lot_size                                                                    AS max_lot_size,\n",
        "            INLINE_VIEW.fixed_lot_size                                                                  AS fixed_lot_size,\n",
        "            INLINE_VIEW.max_stock_level                                                                 AS max_stock_level,\n",
        "            INLINE_VIEW.lot_ordering_cost                                                               AS lot_ordering_cost,\n",
        "            INLINE_VIEW.mrp_time_fence                                                                  AS mrp_time_fence,\n",
        "            INLINE_VIEW.ext_procure_time                                                                AS ext_procure_time,\n",
        "            INLINE_VIEW.internal_mfg_time                                                               AS internal_mfg_time,\n",
        "            INLINE_VIEW.max_storage_days                                                                AS max_storage_days,\n",
        "            COALESCE(INLINE_VIEW.mrp_profile_code, '__NOT_APPLICABLE__')                                AS mrp_profile_code,\n",
        "            COALESCE(INLINE_VIEW.mrp_type_code, '__NOT_APPLICABLE__')                                   AS mrp_type_code,\n",
        "            COALESCE(INLINE_VIEW.mrp_grp_code, '__NOT_APPLICABLE__')                                    AS mrp_grp_code,\n",
        "            COALESCE(INLINE_VIEW.lot_size_code, '__NOT_APPLICABLE__')                                   AS lot_size_code,\n",
        "            INLINE_VIEW.backflush_ind                                                                   AS backflush_ind,\n",
        "            INLINE_VIEW.qa_inspect_ind                                                                  AS qa_inspect_ind,\n",
        "            INLINE_VIEW.repetitive_mfg_ind                                                              AS repetitive_mfg_ind,\n",
        "            INLINE_VIEW.bulk_item_ind                                                                   AS bulk_item_ind,\n",
        "            INLINE_VIEW.forecast_period                                                                 AS forecast_period,\n",
        "            COALESCE(INLINE_VIEW.mfg_uom_code, '__NOT_APPLICABLE__')                                    AS mfg_uom_code,\n",
        "            COALESCE(INLINE_VIEW.issue_uom_code, '__NOT_APPLICABLE__')                                  AS issue_uom_code,\n",
        "            INLINE_VIEW.manufacturing_place                                                             AS manufacturing_place,\n",
        "            COALESCE(INLINE_VIEW.loading_type_code, '__NOT_APPLICABLE__')                               AS loading_type_code,\n",
        "            COALESCE(INLINE_VIEW.int_store_loc_code, '__NOT_APPLICABLE__')                              AS int_store_loc_code,\n",
        "            COALESCE(INLINE_VIEW.ext_store_loc_code, '__NOT_APPLICABLE__')                              AS ext_store_loc_code,\n",
        "            INLINE_VIEW.active_flg                                                                      AS active_flg,\n",
        "            COALESCE(lkp_w_user_d_lkp_w_user_d_cr_1.row_wid, 0)                                          AS created_by_wid,\n",
        "            COALESCE(INLINE_VIEW.row_wid_1, 0)                                                          AS changed_by_wid,\n",
        "            INLINE_VIEW.created_on_dt                                                                   AS created_on_dt,\n",
        "            INLINE_VIEW.changed_on_dt                                                                   AS changed_on_dt,\n",
        "            INLINE_VIEW.aux1_changed_on_dt                                                              AS aux1_changed_on_dt,\n",
        "            INLINE_VIEW.aux2_changed_on_dt                                                              AS aux2_changed_on_dt,\n",
        "            INLINE_VIEW.aux3_changed_on_dt                                                              AS aux3_changed_on_dt,\n",
        "            INLINE_VIEW.aux4_changed_on_dt                                                              AS aux4_changed_on_dt,\n",
        "            INLINE_VIEW.src_eff_from_dt                                                                 AS src_eff_from_dt,\n",
        "            INLINE_VIEW.src_eff_to_dt                                                                   AS src_eff_to_dt,\n",
        "            COALESCE(\n",
        "                INLINE_VIEW.src_eff_from_dt,\n",
        "                to_timestamp(SUBSTRING('${LOW_DATE}', 1, 19), 'yyyy-MM-dd HH:mm:ss')\n",
        "            )                                                                                           AS effective_from_dt,\n",
        "            (CASE WHEN INLINE_VIEW.delete_flg = 'Y' THEN 'Y' ELSE 'N' END)                              AS delete_flg,\n",
        "            INLINE_VIEW.datasource_num_id                                                               AS datasource_num_id,\n",
        "            INLINE_VIEW.integration_id                                                                  AS integration_id,\n",
        "            INLINE_VIEW.tenant_id                                                                       AS tenant_id,\n",
        "            INLINE_VIEW.x_custom                                                                        AS x_custom,\n",
        "            INLINE_VIEW.inv_prod_cat1                                                                   AS inv_prod_cat1,\n",
        "            INLINE_VIEW.inv_prod_cat2                                                                   AS inv_prod_cat2,\n",
        "            INLINE_VIEW.inv_prod_cat3                                                                   AS inv_prod_cat3,\n",
        "            INLINE_VIEW.inv_prod_cat4                                                                   AS inv_prod_cat4,\n",
        "            INLINE_VIEW.inv_prod_cat5                                                                   AS inv_prod_cat5,\n",
        "            INLINE_VIEW.inv_prod_cat6                                                                   AS inv_prod_cat6,\n",
        "            INLINE_VIEW.inv_prod_cat7                                                                   AS inv_prod_cat7,\n",
        "            INLINE_VIEW.inv_prod_cat8                                                                   AS inv_prod_cat8,\n",
        "            INLINE_VIEW.inv_prod_cat9                                                                   AS inv_prod_cat9,\n",
        "            INLINE_VIEW.inv_prod_cat10                                                                  AS inv_prod_cat10,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat1 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat1_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat1_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat2 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat2_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat2_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat3 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat3_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat3_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat4 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat4_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat4_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat5 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat5_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat5_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat6 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat6_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat6_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat7 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat7_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat7_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat8 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat8_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat8_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat9 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat9_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat9_wid,\n",
        "            (CASE WHEN INLINE_VIEW.inv_prod_cat10 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.inv_prod_cat10_row_wid, 0) END)\n",
        "                                                                                                        AS inv_prod_cat10_wid,\n",
        "            COALESCE(INLINE_VIEW.invoiceable_item_flag, 'N')                                            AS invoiceable_item_flag,\n",
        "            COALESCE(INLINE_VIEW.invoice_enabled_flag, 'N')                                             AS invoice_enabled_flag,\n",
        "            COALESCE(INLINE_VIEW.primary_uom_code, '__NOT_APPLICABLE__')                                AS primary_uom_code,\n",
        "            COALESCE(\n",
        "                (\n",
        "                    SELECT\n",
        "                        T.trg_domain_member_code\n",
        "                    FROM\n",
        "                        workspace.prxbi_dw.w_domain_member_map_g AS T\n",
        "                    WHERE\n",
        "                        T.src_domain_code = '${SOURCE_CODE}'\n",
        "                        AND T.src_domain_member_code = COALESCE(INLINE_VIEW.primary_uom_code, '__UNASSIGNED__')\n",
        "                        AND T.src_datasource_num_id IN (INLINE_VIEW.datasource_num_id, 999)\n",
        "                        AND T.trg_domain_code = '${TARGET_CODE}'\n",
        "                ),\n",
        "                (\n",
        "                    SELECT\n",
        "                        T.trg_domain_member_code\n",
        "                    FROM\n",
        "                        workspace.prxbi_dw.w_domain_member_map_g AS T\n",
        "                    WHERE\n",
        "                        T.src_domain_code = '${SOURCE_CODE}'\n",
        "                        AND T.src_domain_member_code = '__ANY__'\n",
        "                        AND T.src_datasource_num_id IN (INLINE_VIEW.datasource_num_id, 999)\n",
        "                        AND T.trg_domain_code = '${TARGET_CODE}'\n",
        "                ),\n",
        "                (\n",
        "                    CASE\n",
        "                        WHEN INLINE_VIEW.primary_uom_code IS NULL THEN COALESCE(\n",
        "                            (\n",
        "                                SELECT\n",
        "                                    T.domain_member_code\n",
        "                                FROM\n",
        "                                    workspace.prxbi_dw.w_domain_member_g AS T\n",
        "                                WHERE\n",
        "                                    T.domain_member_code = '__UNASSIGNED__'\n",
        "                                    AND domain_code = '${TARGET_CODE}'\n",
        "                            ),\n",
        "                            '__ERROR__'\n",
        "                        )\n",
        "                        ELSE (CASE WHEN '${TARGET_CODE}' = 'W_LANGUAGE' THEN '_ERR' ELSE '__ERROR__' END) END\n",
        "                )\n",
        "            )                                                                                           AS c_primary_uom_code,\n",
        "            COALESCE(INLINE_VIEW.unspsc_code, '__NOT_APPLICABLE__')                                     AS unspsc_code,\n",
        "            COALESCE(INLINE_VIEW.inv_prod_cat_unspsc_row_wid, 0)                                        AS unspsc_inv_prod_cat_wid,\n",
        "            INLINE_VIEW.commodity_name                                                                  AS commodity_name,\n",
        "            INLINE_VIEW.commodity_uom_name                                                              AS commodity_uom_name,\n",
        "            INLINE_VIEW.ext_store_loc_name                                                              AS ext_store_loc_name,\n",
        "            INLINE_VIEW.int_store_loc_name                                                              AS int_store_loc_name,\n",
        "            INLINE_VIEW.issue_uom_name                                                                  AS issue_uom_name,\n",
        "            INLINE_VIEW.loading_type_name                                                               AS loading_type_name,\n",
        "            INLINE_VIEW.lot_size_name                                                                   AS lot_size_name,\n",
        "            INLINE_VIEW.mfg_uom_name                                                                    AS mfg_uom_name,\n",
        "            INLINE_VIEW.mrp_grp_name                                                                    AS mrp_grp_name,\n",
        "            INLINE_VIEW.mrp_profile_name                                                                AS mrp_profile_name,\n",
        "            INLINE_VIEW.mrp_type_name                                                                   AS mrp_type_name,\n",
        "            INLINE_VIEW.planner_name                                                                    AS planner_name,\n",
        "            INLINE_VIEW.primary_uom_name                                                                AS primary_uom_name,\n",
        "            INLINE_VIEW.procurement_type_name                                                           AS procurement_type_name,\n",
        "            INLINE_VIEW.profit_center_name                                                              AS profit_center_name,\n",
        "            INLINE_VIEW.spc_proc_type_name                                                              AS spc_proc_type_name,\n",
        "            INLINE_VIEW.status_code                                                                     AS status_code,\n",
        "            INLINE_VIEW.w_status_code                                                                   AS w_status_code,\n",
        "            INLINE_VIEW.product_type_code                                                               AS product_type_code,\n",
        "            INLINE_VIEW.make_buy_ind                                                                    AS make_buy_ind,\n",
        "            INLINE_VIEW.fixed_lead_time                                                                 AS fixed_lead_time,\n",
        "            INLINE_VIEW.variable_lead_time                                                              AS variable_lead_time,\n",
        "            INLINE_VIEW.cumulative_total_lead_time                                                      AS cumulative_total_lead_time,\n",
        "            INLINE_VIEW.preprocessing_lead_time                                                         AS postprocessing_lead_time,\n",
        "            INLINE_VIEW.preprocessing_lead_time                                                         AS preprocessing_lead_time,\n",
        "            INLINE_VIEW.process_quality_enabled_flg                                                     AS process_quality_enabled_flg,\n",
        "            INLINE_VIEW.x_price_sequence                                                                AS x_price_sequence,\n",
        "            INLINE_VIEW.x_organization_name                                                             AS x_organization_name,\n",
        "            INLINE_VIEW.x_product_desc                                                                  AS x_product_desc,\n",
        "            INLINE_VIEW.x_uom_desc                                                                      AS x_uom_desc,\n",
        "            INLINE_VIEW.x_inv_item_flg                                                                  AS x_inv_item_flg,\n",
        "            INLINE_VIEW.x_stock_item_flg                                                                AS x_stock_item_flg,\n",
        "            INLINE_VIEW.x_trans_flg                                                                     AS x_trans_flg,\n",
        "            INLINE_VIEW.x_rev_flg                                                                       AS x_rev_flg,\n",
        "            INLINE_VIEW.x_cost_flg                                                                      AS x_cost_flg,\n",
        "            INLINE_VIEW.x_gcoa_acct                                                                     AS x_gcoa_acct,\n",
        "            INLINE_VIEW.x_gcoa_prod                                                                     AS x_gcoa_prod,\n",
        "            INLINE_VIEW.x_tax_cat                                                                       AS x_tax_cat,\n",
        "            INLINE_VIEW.inventory_org_id                                                                AS organization_id,\n",
        "            INLINE_VIEW.x_gcoa_loc_acct                                                                 AS x_gcoa_loc_acct\n",
        "        FROM\n",
        "            (\n",
        "                SELECT\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_type_name                               AS mrp_type_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_custom                                    AS x_custom,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.src_eff_to_dt                               AS src_eff_to_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.ext_procure_time                            AS ext_procure_time,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_uom_code                          AS commodity_uom_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat10                              AS inv_prod_cat10,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.invoice_enabled_flag                        AS invoice_enabled_flag,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_profile_code                            AS mrp_profile_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.ext_store_loc_name                          AS ext_store_loc_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux3_changed_on_dt                          AS aux3_changed_on_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.min_lot_size                                AS min_lot_size,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.reorder_point                               AS reorder_point,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.bulk_item_ind                               AS bulk_item_ind,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.spc_proc_type_name                          AS spc_proc_type_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_code                              AS commodity_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.repetitive_mfg_ind                          AS repetitive_mfg_ind,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.loading_type_code                           AS loading_type_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.profit_center_name                          AS profit_center_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_by_id                               AS created_by_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.buyer_code                                  AS buyer_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.int_store_loc_name                          AS int_store_loc_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_name                              AS commodity_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.planner_name                                AS planner_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.max_storage_days                            AS max_storage_days,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.planner_code                                AS planner_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.max_lot_size                                AS max_lot_size,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_grp_code                                AS mrp_grp_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.lot_ordering_cost                           AS lot_ordering_cost,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.internal_mfg_time                           AS internal_mfg_time,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.manufacturing_place                         AS manufacturing_place,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.procurement_type_name                       AS procurement_type_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.plant_loc_id                                AS plant_loc_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_by_id                               AS changed_by_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.backflush_ind                               AS backflush_ind,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux2_changed_on_dt                          AS aux2_changed_on_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id                           AS datasource_num_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.spc_proc_type_code                          AS spc_proc_type_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.lot_size_code                               AS lot_size_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_on_dt                               AS changed_on_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_id                                  AS product_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.primary_uom_code                            AS primary_uom_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_num                                 AS product_num,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.forecast_period                             AS forecast_period,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux1_changed_on_dt                          AS aux1_changed_on_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.buyer_name                                  AS buyer_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.fixed_lot_size                              AS fixed_lot_size,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_grp_name                                AS mrp_grp_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.procurement_type_code                       AS procurement_type_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.primary_uom_name                            AS primary_uom_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.max_stock_level                             AS max_stock_level,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.safety_stock_level                          AS safety_stock_level,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.int_store_loc_code                          AS int_store_loc_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.ext_store_loc_code                          AS ext_store_loc_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.issue_uom_name                              AS issue_uom_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.abc_ind                                     AS abc_ind,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.profit_center_num                           AS profit_center_num,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt                               AS created_on_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.unspsc_code                                 AS unspsc_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat1                               AS inv_prod_cat1,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat3                               AS inv_prod_cat3,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat2                               AS inv_prod_cat2,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_uom_name                          AS commodity_uom_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat5                               AS inv_prod_cat5,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat4                               AS inv_prod_cat4,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mfg_uom_name                                AS mfg_uom_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat7                               AS inv_prod_cat7,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat6                               AS inv_prod_cat6,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat9                               AS inv_prod_cat9,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat8                               AS inv_prod_cat8,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux4_changed_on_dt                          AS aux4_changed_on_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.lot_size_name                               AS lot_size_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.tenant_id                                   AS tenant_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.invoiceable_item_flag                       AS invoiceable_item_flag,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inventory_org_id                            AS inventory_org_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.src_eff_from_dt                             AS src_eff_from_dt,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.integration_id                              AS integration_id,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.issue_uom_code                              AS issue_uom_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.delete_flg                                  AS delete_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_type_code                               AS mrp_type_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.active_flg                                  AS active_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_time_fence                              AS mrp_time_fence,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.qa_inspect_ind                              AS qa_inspect_ind,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mfg_uom_code                                AS mfg_uom_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.loading_type_name                           AS loading_type_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_profile_name                            AS mrp_profile_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat1_row_wid                       AS inv_prod_cat1_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat2_row_wid                       AS inv_prod_cat2_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat3_row_wid                       AS inv_prod_cat3_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat4_row_wid                       AS inv_prod_cat4_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat5_row_wid                       AS inv_prod_cat5_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat6_row_wid                       AS inv_prod_cat6_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat7_row_wid                       AS inv_prod_cat7_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat8_row_wid                       AS inv_prod_cat8_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat_unspsc_row_wid                 AS inv_prod_cat_unspsc_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat9_row_wid                       AS inv_prod_cat9_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat10_row_wid                      AS inv_prod_cat10_row_wid,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.cumulative_total_lead_time                  AS cumulative_total_lead_time,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.fixed_lead_time                             AS fixed_lead_time,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.w_status_code                               AS w_status_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.preprocessing_lead_time                     AS preprocessing_lead_time,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.status_code                                 AS status_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.variable_lead_time                          AS variable_lead_time,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.make_buy_ind                                AS make_buy_ind,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.process_quality_enabled_flg                 AS process_quality_enabled_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_type_code                           AS product_type_code,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_price_sequence                            AS x_price_sequence,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_organization_name                         AS x_organization_name,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_product_desc                              AS x_product_desc,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_uom_desc                                  AS x_uom_desc,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_inv_item_flg                              AS x_inv_item_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_stock_item_flg                            AS x_stock_item_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_trans_flg                                 AS x_trans_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_rev_flg                                   AS x_rev_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_cost_flg                                  AS x_cost_flg,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_gcoa_acct                                 AS x_gcoa_acct,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_gcoa_prod                                 AS x_gcoa_prod,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_tax_cat                                   AS x_tax_cat,\n",
        "                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_gcoa_loc_acct                             AS x_gcoa_loc_acct,\n",
        "                    LKP_W_BUSN_LOCATION_D_LKP_W_BU.datasource_num_id                           AS datasource_num_id_1,\n",
        "                    LKP_W_BUSN_LOCATION_D_LKP_W_BU.row_wid                                     AS row_wid,\n",
        "                    LKP_W_BUSN_LOCATION_D_LKP_W_BU.integration_id                              AS integration_id_1,\n",
        "                    LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_to_dt                             AS effective_to_dt,\n",
        "                    LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_from_dt                           AS effective_from_dt,\n",
        "                    LKP_W_INT_ORG_D_INVENTORY.effective_from_dt                                AS effective_from_dt_1,\n",
        "                    LKP_W_INT_ORG_D_INVENTORY.effective_to_dt                                  AS effective_to_dt_1,\n",
        "                    LKP_W_INT_ORG_D_INVENTORY.datasource_num_id                                AS datasource_num_id_2,\n",
        "                    LKP_W_INT_ORG_D_INVENTORY.integration_id                                   AS integration_id_2,\n",
        "                    LKP_W_INT_ORG_D_INVENTORY.scd1_wid                                         AS scd1_wid,\n",
        "                    LKP_W_PRODUCT_D_PRODUCT_WID.effective_from_dt                              AS effective_from_dt_2,\n",
        "                    LKP_W_PRODUCT_D_PRODUCT_WID.effective_to_dt                                AS effective_to_dt_2,\n",
        "                    LKP_W_PRODUCT_D_PRODUCT_WID.datasource_num_id                              AS datasource_num_id_3,\n",
        "                    LKP_W_PRODUCT_D_PRODUCT_WID.integration_id                                 AS integration_id_3,\n",
        "                    LKP_W_PRODUCT_D_PRODUCT_WID.scd1_wid                                       AS scd1_wid_1,\n",
        "                    LKP_W_USER_D_LKP_W_USER_D_CHAN.datasource_num_id                           AS datasource_num_id_4,\n",
        "                    LKP_W_USER_D_LKP_W_USER_D_CHAN.row_wid                                     AS row_wid_1,\n",
        "                    LKP_W_USER_D_LKP_W_USER_D_CHAN.integration_id                              AS integration_id_4,\n",
        "                    LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_to_dt                             AS effective_to_dt_3,\n",
        "                    LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_from_dt                           AS effective_from_dt_3\n",
        "                FROM\n",
        "                    (\n",
        "                        SELECT\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_type_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_custom,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.src_eff_to_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.ext_procure_time,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.commodity_uom_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat10,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.invoice_enabled_flag,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_profile_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.ext_store_loc_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.aux3_changed_on_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.min_lot_size,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.reorder_point,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.bulk_item_ind,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.spc_proc_type_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.commodity_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.repetitive_mfg_ind,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.loading_type_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.profit_center_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.created_by_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.buyer_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.int_store_loc_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.commodity_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.planner_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.max_storage_days,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.planner_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.max_lot_size,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_grp_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.lot_ordering_cost,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.internal_mfg_time,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.manufacturing_place,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.procurement_type_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.plant_loc_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.changed_by_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.backflush_ind,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.aux2_changed_on_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.spc_proc_type_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.lot_size_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.changed_on_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.product_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.primary_uom_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.product_num,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.forecast_period,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.aux1_changed_on_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.buyer_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.fixed_lot_size,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_grp_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.procurement_type_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.primary_uom_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.max_stock_level,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.safety_stock_level,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.int_store_loc_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.ext_store_loc_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.issue_uom_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.abc_ind,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.profit_center_num,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.created_on_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.unspsc_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat1,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat3,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat2,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.commodity_uom_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat5,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat4,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mfg_uom_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat7,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat6,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat9,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat8,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.aux4_changed_on_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.lot_size_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.tenant_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.invoiceable_item_flag,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.inventory_org_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.src_eff_from_dt,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.integration_id,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.issue_uom_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.delete_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_type_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.active_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_time_fence,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.qa_inspect_ind,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mfg_uom_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.loading_type_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.mrp_profile_name,\n",
        "                            W_PROD_CAT_DH1_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat1_row_wid,\n",
        "                            W_PROD_CAT_DH2_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat2_row_wid,\n",
        "                            W_PROD_CAT_DH3_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat3_row_wid,\n",
        "                            W_PROD_CAT_DH4_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat4_row_wid,\n",
        "                            W_PROD_CAT_DH5_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat5_row_wid,\n",
        "                            W_PROD_CAT_DH6_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat6_row_wid,\n",
        "                            W_PROD_CAT_DH7_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat7_row_wid,\n",
        "                            W_PROD_CAT_DH8_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat8_row_wid,\n",
        "                            W_PROD_CAT_DH_UNSPSC_SQ_W_INVE.row_wid        AS inv_prod_cat_unspsc_row_wid,\n",
        "                            W_PROD_CAT_DH9_SQ_W_INVENTORY_.row_wid        AS inv_prod_cat9_row_wid,\n",
        "                            W_PROD_CAT_DH10_SQ_W_INVENTORY.row_wid        AS inv_prod_cat10_row_wid,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.cumulative_total_lead_time,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.fixed_lead_time,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.w_status_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.preprocessing_lead_time,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.status_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.variable_lead_time,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.make_buy_ind,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.process_quality_enabled_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.product_type_code,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_price_sequence,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_organization_name,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_product_desc,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_uom_desc,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_inv_item_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_stock_item_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_trans_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_rev_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_cost_flg,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_gcoa_acct,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_gcoa_prod,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_tax_cat,\n",
        "                            W_INVENTORY_PRODUCT_DS_SQ_W_IN.x_gcoa_loc_acct\n",
        "                        FROM\n",
        "                            (\n",
        "                                (\n",
        "                                    (\n",
        "                                        (\n",
        "                                            (\n",
        "                                                (\n",
        "                                                    (\n",
        "                                                        (\n",
        "                                                            (\n",
        "                                                                (\n",
        "                                                                    (\n",
        "                                                                        workspace.prxbi_dw.w_inventory_product_ds AS W_INVENTORY_PRODUCT_DS_SQ_W_IN\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH1_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat1 = W_PROD_CAT_DH1_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH1_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH2_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat2 = W_PROD_CAT_DH2_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH2_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH3_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat3 = W_PROD_CAT_DH3_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH3_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH4_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat4 = W_PROD_CAT_DH4_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH4_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH5_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat5 = W_PROD_CAT_DH5_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH5_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH6_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat6 = W_PROD_CAT_DH6_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH6_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH7_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat7 = W_PROD_CAT_DH7_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH7_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH8_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat8 = W_PROD_CAT_DH8_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH8_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH9_SQ_W_INVENTORY_\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat9 = W_PROD_CAT_DH9_SQ_W_INVENTORY_.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH9_SQ_W_INVENTORY_.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH10_SQ_W_INVENTORY\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.inv_prod_cat10 = W_PROD_CAT_DH10_SQ_W_INVENTORY.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH10_SQ_W_INVENTORY.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH_UNSPSC_SQ_W_INVE\n",
        "                                                                            ON W_INVENTORY_PRODUCT_DS_SQ_W_IN.unspsc_code = W_PROD_CAT_DH_UNSPSC_SQ_W_INVE.integration_id\n",
        "                                                                            AND W_INVENTORY_PRODUCT_DS_SQ_W_IN.datasource_num_id = W_PROD_CAT_DH_UNSPSC_SQ_W_INVE.datasource_num_id\n",
        "                                                                        )\n",
        "                                                                    WHERE 1 = 1\n",
        "                                                                ) AS SQ_W_INVENTORY_PRODUCT_DS_SQ_W\n",
        "                                                                LEFT OUTER JOIN (\n",
        "                                                                    SELECT\n",
        "                                                                        W_BUSN_LOCATION_D_LKP_W_BUSN_L.datasource_num_id AS datasource_num_id,\n",
        "                                                                        W_BUSN_LOCATION_D_LKP_W_BUSN_L.row_wid          AS row_wid,\n",
        "                                                                        W_BUSN_LOCATION_D_LKP_W_BUSN_L.integration_id    AS integration_id,\n",
        "                                                                        W_BUSN_LOCATION_D_LKP_W_BUSN_L.effective_to_dt   AS effective_to_dt,\n",
        "                                                                        W_BUSN_LOCATION_D_LKP_W_BUSN_L.effective_from_dt AS effective_from_dt\n",
        "                                                                    FROM\n",
        "                                                                        workspace.prxbi_dw.w_busn_location_d AS W_BUSN_LOCATION_D_LKP_W_BUSN_L\n",
        "                                                                    WHERE 1 = 1\n",
        "                                                                ) AS LKP_W_BUSN_LOCATION_D_LKP_W_BU\n",
        "                                                                    ON SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_BUSN_LOCATION_D_LKP_W_BU.datasource_num_id\n",
        "                                                                    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.plant_loc_id = LKP_W_BUSN_LOCATION_D_LKP_W_BU.integration_id\n",
        "                                                                    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt >= LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_from_dt\n",
        "                                                                    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt < LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_to_dt\n",
        "                                                            )\n",
        "                                                            LEFT OUTER JOIN workspace.prxbi_dw.w_int_org_d AS LKP_W_INT_ORG_D_INVENTORY\n",
        "                                                                ON SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_INT_ORG_D_INVENTORY.datasource_num_id\n",
        "                                                                AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inventory_org_id = LKP_W_INT_ORG_D_INVENTORY.integration_id\n",
        "                                                                AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt >= LKP_W_INT_ORG_D_INVENTORY.effective_from_dt\n",
        "                                                                AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt < LKP_W_INT_ORG_D_INVENTORY.effective_to_dt\n",
        "                                                        )\n",
        "                                                        LEFT OUTER JOIN workspace.prxbi_dw.w_product_d AS LKP_W_PRODUCT_D_PRODUCT_WID\n",
        "                                                            ON SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_PRODUCT_D_PRODUCT_WID.datasource_num_id\n",
        "                                                            AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_id = LKP_W_PRODUCT_D_PRODUCT_WID.integration_id\n",
        "                                                            AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt >= LKP_W_PRODUCT_D_PRODUCT_WID.effective_from_dt\n",
        "                                                            AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt < LKP_W_PRODUCT_D_PRODUCT_WID.effective_to_dt\n",
        "                                                    )\n",
        "                                                    LEFT OUTER JOIN (\n",
        "                                                        SELECT\n",
        "                                                            W_USER_D_LKP_W_USER_D_CHANGED_.datasource_num_id AS datasource_num_id,\n",
        "                                                            W_USER_D_LKP_W_USER_D_CHANGED_.row_wid          AS row_wid,\n",
        "                                                            W_USER_D_LKP_W_USER_D_CHANGED_.integration_id    AS integration_id,\n",
        "                                                            W_USER_D_LKP_W_USER_D_CHANGED_.effective_to_dt   AS effective_to_dt,\n",
        "                                                            W_USER_D_LKP_W_USER_D_CHANGED_.effective_from_dt AS effective_from_dt\n",
        "                                                        FROM\n",
        "                                                            workspace.prxbi_dw.w_user_d AS W_USER_D_LKP_W_USER_D_CHANGED_\n",
        "                                                        WHERE 1 = 1 AND (W_USER_D_LKP_W_USER_D_CHANGED_.delete_flg = 'N')\n",
        "                                                    ) AS LKP_W_USER_D_LKP_W_USER_D_CHAN\n",
        "                                                        ON SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_USER_D_LKP_W_USER_D_CHAN.datasource_num_id\n",
        "                                                        AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_by_id = LKP_W_USER_D_LKP_W_USER_D_CHAN.integration_id\n",
        "                                                        AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_on_dt >= LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_from_dt\n",
        "                                                        AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_on_dt < LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_to_dt\n",
        "                                                ) AS INLINE_VIEW\n",
        "                                                LEFT OUTER JOIN (\n",
        "                                                    SELECT\n",
        "                                                        LKP_W_USER_D_LKP_W_USER_D_CREA.datasource_num_id AS datasource_num_id,\n",
        "                                                        LKP_W_USER_D_LKP_W_USER_D_CREA.row_wid          AS row_wid,\n",
        "                                                        LKP_W_USER_D_LKP_W_USER_D_CREA.integration_id    AS integration_id,\n",
        "                                                        LKP_W_USER_D_LKP_W_USER_D_CREA.effective_to_dt   AS effective_to_dt,\n",
        "                                                        LKP_W_USER_D_LKP_W_USER_D_CREA.effective_from_dt AS effective_from_dt\n",
        "                                                    FROM\n",
        "                                                        (\n",
        "                                                            SELECT\n",
        "                                                                W_USER_D_LKP_W_USER_D_CREATED_.datasource_num_id AS datasource_num_id,\n",
        "                                                                W_USER_D_LKP_W_USER_D_CREATED_.row_wid          AS row_wid,\n",
        "                                                                W_USER_D_LKP_